# Part 2: Routing banking support tickets with fine-tuned transformers
**Data:** Banking77 (PolyAI, CC-BY-4.0): 13,083 real customer queries to a banking app, labelled into **77 fine-grained intents** (e.g. *card_arrival*, *pending_top_up*, *exchange_rate*).

**Business question:** can incoming tickets be routed to the right queue automatically, and how many can be routed safely without a human?

| Model | Idea |
|---|---|
| TF-IDF + logistic regression | classic baseline |
| MiniLM sentence embeddings + logistic regression | frozen 🤗 encoder, light classifier |
| DistilBERT fine-tuned | full fine-tuning with the 🤗 `Trainer`, 2 epochs on CPU (a deliberately small compute budget) |

Training runs once in `src/intent.py` (cached predictions); this notebook analyses the results.

In [ ]:
import sys, json, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from IPython.display import Image, display
from intent import *
import charts
train, test, names = load_banking77()
print(f'{len(train):,} train · {len(test):,} test · {len(names)} intents')
test.sample(5, random_state=1).assign(intent=lambda d: d.label.map(dict(enumerate(names))))

In [ ]:
res = pd.read_csv('../reports/intent_model_comparison.csv')
charts.intent_models(res)
display(res.round(4))
display(Image('../images/intent_models.png'))

In [ ]:
hist = pd.DataFrame(json.loads((CACHE / 'distilbert_history.json').read_text()))
hist[['epoch', 'loss']].iloc[::3].round(3).T

**Reading the result:** with only 2 epochs on CPU, DistilBERT is under-trained (loss still falling), and a *frozen* MiniLM encoder plus logistic regression wins at about a ninth of the compute. The Banking77 paper reports about 93% for fully fine-tuned BERT, so the fine-tune should overtake with more epochs on a GPU. On a CPU budget, embeddings + a linear model are the pragmatic choice.

## Safe automation: route only confident predictions (best model: MiniLM + LogReg)

In [ ]:
P = np.load(CACHE / 'banking77_test_probs.npz')
y = P['y']
curves = {'DistilBERT fine-tuned': routing_curve(y, P['distilbert']), 'MiniLM + LogReg': routing_curve(y, P['minilm']),
          'TF-IDF + LogReg': routing_curve(y, P['tfidf'])}
charts.routing_chart(curves)
c = curves['MiniLM + LogReg']
display(c.round({"threshold": 2, "auto_routed_pct": 1, "accuracy_on_auto_routed_pct": 1}))
display(Image('../images/routing_tradeoff.png'))

In [ ]:
op = c[c.accuracy_on_auto_routed_pct >= 98].iloc[0]
print(f"At confidence ≥ {op.threshold:.2f}, the MiniLM model auto-routes {op.auto_routed_pct:.0f}% of tickets at {op.accuracy_on_auto_routed_pct:.1f}% accuracy; "
      f"the remaining {100 - op.auto_routed_pct:.0f}% go to human triage.")

## Where does it still get confused?

In [ ]:
top_confusions(y, P['minilm'], names, 10)

Most confusions are between genuinely overlapping intents (e.g. *pending* vs *failed* transfers or top-ups). These are candidates for merging queues or adding a clarifying question in the chatbot.